In [31]:
import s3fs, re, numpy as np, pandas as pd
import pyarrow.parquet as pq
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings('ignore')
from s3_credentials import S3_ACCESS_KEY, S3_SECRET_KEY
OPTS={"key":S3_ACCESS_KEY,"secret":S3_SECRET_KEY,
      "client_kwargs":{"endpoint_url":"https://iseadocker.isea.rwth-aachen.de:9000","region_name":"us-east-1"},
      "config_kwargs":{"s3":{"addressing_style":"path"},"signature_version":"s3v4"}}
fs=s3fs.S3FileSystem(**OPTS)
RAWBASE="projects/j8005-metabatt/Metabatt/A123"
NOMINAL=1.1
bid="008"
BASE=f"{RAWBASE}/METABatt_A123_APR18650M1B_{bid}"
aging=sorted([f for f in fs.ls(BASE) if 'Aging' in f and 'Cyc' in f])
print(f"aging文件数: {len(aging)}")

# ===== 第1部分:跨生命周期密集采样(每个文件取多个放电片段) =====
def get_segments(df):
    df=df.copy(); df['Zeit']=pd.to_datetime(df['Zeit'])
    is_dch=((df['Zustand']=='DCH')&(df['Strom']<-1)).values
    ch=np.diff(is_dch.astype(int)); st=np.where(ch==1)[0]+1; en=np.where(ch==-1)[0]+1
    if len(is_dch)>0 and is_dch[0]: st=np.r_[0,st]
    if len(en)<len(st): en=np.r_[en,len(df)]
    out=[]
    for s,e in zip(st,en):
        seg=df.iloc[s:e]
        if len(seg)>=30: out.append(seg)
    return out

print("\n=== 第1部分:跨生命周期 电流/容量/时长 密集统计 ===")
rows=[]; cum_efc=0.0
for fi,f in enumerate(aging):
    with fs.open(f) as fh:
        pf=pq.ParquetFile(fh)
        df=next(pf.iter_batches(batch_size=500000,columns=['Zeit','Spannung','Strom','Zustand'])).to_pandas()
    segs=get_segments(df)
    for seg in segs[:20]:   # 每个文件最多取20个片段
        V=seg['Spannung'].values; I=seg['Strom'].values
        tt=(seg['Zeit']-seg['Zeit'].iloc[0]).dt.total_seconds().values
        dur=tt[-1]-tt[0]
        Q=np.sum(np.abs(I)*np.diff(tt,prepend=tt[0])/3600)
        rows.append({'file':fi,'I_mean':I.mean(),'I_max':I.min(),'dur':dur,'cap':Q,
                     'v_span':V[0]-V[-1],'v_start':V[0],'v_end':V[-1]})
    # 估EFC
    t=df['Zeit'].diff().dt.total_seconds().fillna(0).values if 'Zeit' in df else np.zeros(len(df))
    cum_efc+= (np.abs(df['Strom'].values)*np.clip(pd.to_datetime(df['Zeit']).diff().dt.total_seconds().fillna(0).values,0,10)/3600).sum()/NOMINAL
    print(f"  [{fi+1}/{len(aging)}] EFC≈{cum_efc:.0f}, 片段累计{len(rows)}")

life=pd.DataFrame(rows)
life['seq']=range(len(life))
life.to_csv("008_current_lifecycle.csv",index=False)

print(f"\n电流随时间(文件序): 首均值{life['I_mean'].iloc[:10].mean():.3f}A → 末均值{life['I_mean'].iloc[-10:].mean():.3f}A")
print(f"时长: 首{life['dur'].iloc[:10].mean():.0f}s → 末{life['dur'].iloc[-10:].mean():.0f}s")
print(f"时长是否恒定: min{life['dur'].min():.0f} max{life['dur'].max():.0f} std{life['dur'].std():.1f}")

# 画跨生命周期变化
fig,axes=plt.subplots(2,2,figsize=(12,8))
axes[0,0].plot(life['seq'],life['I_mean'].abs()); axes[0,0].set_title('电流均值 vs 循环序'); axes[0,0].set_ylabel('|I| (A)')
axes[0,1].plot(life['seq'],life['cap']); axes[0,1].set_title('放电容量 vs 循环序'); axes[0,1].set_ylabel('Cap (Ah)')
axes[1,0].plot(life['seq'],life['dur']); axes[1,0].set_title('放电时长 vs 循环序'); axes[1,0].set_ylabel('时长 (s)')
axes[1,1].plot(life['seq'],life['v_span']); axes[1,1].set_title('电压跨度 vs 循环序'); axes[1,1].set_ylabel('V_span (V)')
plt.tight_layout(); plt.savefig('008_lifecycle_trends.png',dpi=100); print("\n图1已存: 008_lifecycle_trends.png")

# ===== 第2部分:早/中/晚期各取一次放电,画内部I和V随时间 =====
print("\n=== 第2部分:单次放电内部 I/V 随时间(早/中/晚) ===")
fig,axes=plt.subplots(1,2,figsize=(14,5))
for label,idx,color in [('早期',0,'b'),('中期',len(aging)//2,'g'),('晚期',len(aging)-1,'r')]:
    with fs.open(aging[idx]) as fh:
        pf=pq.ParquetFile(fh)
        df=next(pf.iter_batches(batch_size=500000,columns=['Zeit','Spannung','Strom','Zustand'])).to_pandas()
    segs=get_segments(df)
    if not segs: continue
    seg=segs[0]
    tt=(pd.to_datetime(seg['Zeit'])-pd.to_datetime(seg['Zeit']).iloc[0]).dt.total_seconds().values
    axes[0].plot(tt,seg['Strom'].values,color,label=f'{label}(文件{idx})')
    axes[1].plot(tt,seg['Spannung'].values,color,label=f'{label}(文件{idx})')
axes[0].set_title('单次放电 电流vs时间'); axes[0].set_xlabel('秒'); axes[0].set_ylabel('I(A)'); axes[0].legend()
axes[1].set_title('单次放电 电压vs时间'); axes[1].set_xlabel('秒'); axes[1].set_ylabel('V(V)'); axes[1].legend()
plt.tight_layout(); plt.savefig('008_single_discharge.png',dpi=100); print("图2已存: 008_single_discharge.png")
print("\n看图2电流曲线:若一次放电内I恒定→恒流;若I在变→恒功率/受限")

aging文件数: 21

=== 第1部分:跨生命周期 电流/容量/时长 密集统计 ===
  [1/21] EFC≈112, 片段累计20
  [2/21] EFC≈176, 片段累计40
  [3/21] EFC≈289, 片段累计60
  [4/21] EFC≈409, 片段累计80
  [5/21] EFC≈524, 片段累计100
  [6/21] EFC≈639, 片段累计120
  [7/21] EFC≈750, 片段累计140
  [8/21] EFC≈859, 片段累计160
  [9/21] EFC≈966, 片段累计180
  [10/21] EFC≈1047, 片段累计200
  [11/21] EFC≈1137, 片段累计220
  [12/21] EFC≈1221, 片段累计240
  [13/21] EFC≈1323, 片段累计260
  [14/21] EFC≈1423, 片段累计280
  [15/21] EFC≈1523, 片段累计300
  [16/21] EFC≈1621, 片段累计320
  [17/21] EFC≈1714, 片段累计340
  [18/21] EFC≈1808, 片段累计360
  [19/21] EFC≈1903, 片段累计380
  [20/21] EFC≈1997, 片段累计400
  [21/21] EFC≈2082, 片段累计420

电流随时间(文件序): 首均值-3.558A → 末均值-2.641A
时长: 首720s → 末720s
时长是否恒定: min720 max720 std0.0

图1已存: 008_lifecycle_trends.png

=== 第2部分:单次放电内部 I/V 随时间(早/中/晚) ===
图2已存: 008_single_discharge.png

看图2电流曲线:若一次放电内I恒定→恒流;若I在变→恒功率/受限


In [33]:
import s3fs, re, numpy as np, pandas as pd
import pyarrow.parquet as pq
import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings('ignore')
from s3_credentials import S3_ACCESS_KEY, S3_SECRET_KEY
OPTS={"key":S3_ACCESS_KEY,"secret":S3_SECRET_KEY,
      "client_kwargs":{"endpoint_url":"https://iseadocker.isea.rwth-aachen.de:9000","region_name":"us-east-1"},
      "config_kwargs":{"s3":{"addressing_style":"path"},"signature_version":"s3v4"}}
RAWBASE="projects/j8005-metabatt/Metabatt/A123"
fs=s3fs.S3FileSystem(**OPTS)
bid="008"
BASE=f"{RAWBASE}/METABatt_A123_APR18650M1B_{bid}"
aging=sorted([f for f in fs.ls(BASE) if 'Aging' in f and 'Cyc' in f])

def get_charge_segments(df):
    df=df.copy(); df['Zeit']=pd.to_datetime(df['Zeit'])
    is_cha=(df['Zustand']=='CHA').values   # 充电
    ch=np.diff(is_cha.astype(int)); st=np.where(ch==1)[0]+1; en=np.where(ch==-1)[0]+1
    if len(is_cha)>0 and is_cha[0]: st=np.r_[0,st]
    if len(en)<len(st): en=np.r_[en,len(df)]
    return [df.iloc[s:e] for s,e in zip(st,en) if e-s>=30]

# 早中晚各取一个充电段,画电流电压随时间
print("=== 充电段机制探查 ===")
fig,axes=plt.subplots(1,2,figsize=(14,5))
for label,idx,c in [('早期',0,'b'),('中期',len(aging)//2,'g'),('晚期',len(aging)-1,'r')]:
    with fs.open(aging[idx]) as fh:
        df=next(pq.ParquetFile(fh).iter_batches(batch_size=500000,
                columns=['Zeit','Spannung','Strom','Zustand']).__iter__()).to_pandas()
    segs=get_charge_segments(df)
    if not segs: print(f"{label}: 无充电段"); continue
    seg=segs[0]
    tt=(pd.to_datetime(seg['Zeit'])-pd.to_datetime(seg['Zeit']).iloc[0]).dt.total_seconds().values
    I=seg['Strom'].values; V=seg['Spannung'].values
    axes[0].plot(tt,I,c,label=f'{label}')
    axes[1].plot(tt,V,c,label=f'{label}')
    # 判断CC-CV:电流恒定段 vs 电压恒定段
    print(f"{label}(文件{idx}): {len(seg)}行 时长{tt[-1]:.0f}s")
    print(f"   电流 {I.min():.2f}~{I.max():.2f}A (均值{I.mean():.2f}, std{I.std():.3f})")
    print(f"   电压 {V.min():.3f}~{V.max():.3f}V")
    # CC段=电流稳定的部分,CV段=电压稳定的部分
    i_stable=np.sum(np.abs(I-I[len(I)//4])<0.05)/len(I)*100
    print(f"   电流稳定占比≈{i_stable:.0f}% (高=有明显CC段)")
axes[0].set_title('充电 电流vs时间'); axes[0].set_xlabel('s'); axes[0].set_ylabel('I(A)'); axes[0].legend()
axes[1].set_title('充电 电压vs时间'); axes[1].set_xlabel('s'); axes[1].set_ylabel('V(V)'); axes[1].legend()
plt.tight_layout(); plt.savefig('008_charge_mechanism.png',dpi=100)
print("\n图已存: 008_charge_mechanism.png")
print("""
看图判断:
- 电流先平(CC)后降(CV) + 电压先升后平 → 标准CC-CV
- 三期电流水平是否相同 → 相同则充电电流固定(无泄露,好!);不同则也绑定容量
- 电压形状随老化变化 → 老化信息
""")

=== 充电段机制探查 ===
早期(文件0): 2414行 时长1167s
   电流 3.56~3.56A (均值3.56, std0.000)
   电压 2.800~3.600V
   电流稳定占比≈100% (高=有明显CC段)
中期(文件10): 2246行 时长1086s
   电流 2.93~2.93A (均值2.93, std0.000)
   电压 2.849~3.600V
   电流稳定占比≈100% (高=有明显CC段)
晚期(文件20): 2211行 时长1064s
   电流 2.64~2.64A (均值2.64, std0.000)
   电压 2.765~3.600V
   电流稳定占比≈100% (高=有明显CC段)

图已存: 008_charge_mechanism.png

看图判断:
- 电流先平(CC)后降(CV) + 电压先升后平 → 标准CC-CV
- 三期电流水平是否相同 → 相同则充电电流固定(无泄露,好!);不同则也绑定容量
- 电压形状随老化变化 → 老化信息



In [35]:
import s3fs, re, numpy as np, pandas as pd
import pyarrow.parquet as pq
import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings('ignore')
from s3_credentials import S3_ACCESS_KEY, S3_SECRET_KEY
OPTS={"key":S3_ACCESS_KEY,"secret":S3_SECRET_KEY,
      "client_kwargs":{"endpoint_url":"https://iseadocker.isea.rwth-aachen.de:9000","region_name":"us-east-1"},
      "config_kwargs":{"s3":{"addressing_style":"path"},"signature_version":"s3v4"}}
RAWBASE="projects/j8005-metabatt/Metabatt/A123"
fs=s3fs.S3FileSystem(**OPTS)
bid="008"
BASE=f"{RAWBASE}/METABatt_A123_APR18650M1B_{bid}"
aging=sorted([f for f in fs.ls(BASE) if 'Aging' in f and 'Cyc' in f])

# 读最后一个文件(最晚期)
f=aging[-1]
print(f"=== 晚期文件: {f.split('/')[-1][:60]} ===")
with fs.open(f) as fh:
    df=next(pq.ParquetFile(fh).iter_batches(batch_size=500000,
            columns=['Zeit','Spannung','Strom','Zustand']).__iter__()).to_pandas()
df['Zeit']=pd.to_datetime(df['Zeit']); df=df.sort_values('Zeit').reset_index(drop=True)
print(f"行数:{len(df)}, 状态分布:{df['Zustand'].value_counts().to_dict()}")

# 切出所有片段(CHA和DCH),按顺序看充放是否配对
def segments(df,state,cur_sign):
    m=((df['Zustand']==state)&(np.sign(df['Strom'])==cur_sign)).values
    ch=np.diff(m.astype(int)); st=np.where(ch==1)[0]+1; en=np.where(ch==-1)[0]+1
    if len(m)>0 and m[0]: st=np.r_[0,st]
    if len(en)<len(st): en=np.r_[en,len(df)]
    return [(s,e) for s,e in zip(st,en) if e-s>=30]

# 按时间顺序,交替列出充电和放电片段,看配对
print("\n=== 晚期 充放电片段按时间顺序(看是否充放配对闭环) ===")
all_segs=[]
for s,e in segments(df,'CHA',1): all_segs.append(('CHA',s,e))
for s,e in segments(df,'DCH',-1): all_segs.append(('DCH',s,e))
all_segs.sort(key=lambda x:x[1])  # 按起始位置排序=时间顺序

print(f"{'类型':>5}{'时长s':>8}{'V起':>8}{'V止':>8}{'V跨度':>8}{'容量Ah':>10}{'电流A':>8}")
for typ,s,e in all_segs[:12]:  # 看前12个(几个充放循环)
    seg=df.iloc[s:e]
    V=seg['Spannung'].values; I=seg['Strom'].values
    tt=(seg['Zeit']-seg['Zeit'].iloc[0]).dt.total_seconds().values
    Q=np.sum(np.abs(I)*np.diff(tt,prepend=tt[0])/3600)
    print(f"{typ:>5}{tt[-1]:>8.0f}{V[0]:>8.3f}{V[-1]:>8.3f}{abs(V[-1]-V[0]):>8.3f}{Q:>10.4f}{I.mean():>8.2f}")

# 画晚期一个完整充放循环
print("\n画晚期一个充放循环...")
fig,ax=plt.subplots(1,2,figsize=(14,5))
# 取第一个CHA和它后面第一个DCH
cha=[x for x in all_segs if x[0]=='CHA'][0]
dch=[x for x in all_segs if x[0]=='DCH'][0]
for typ,s,e in [cha,dch]:
    seg=df.iloc[s:e]
    tt=(seg['Zeit']-seg['Zeit'].iloc[0]).dt.total_seconds().values
    ax[0].plot(tt,seg['Spannung'].values,label=f'{typ}')
    ax[1].plot(tt,seg['Strom'].values,label=f'{typ}')
ax[0].set_title('晚期 电压vs时间'); ax[0].set_xlabel('s'); ax[0].set_ylabel('V'); ax[0].legend()
ax[1].set_title('晚期 电流vs时间'); ax[1].set_xlabel('s'); ax[1].set_ylabel('I'); ax[1].legend()
plt.tight_layout(); plt.savefig('008_late_cycle.png',dpi=100)
print("图已存: 008_late_cycle.png")

# 对比:早期同样dump
print("\n=== 早期对比(第一个文件) ===")
with fs.open(aging[0]) as fh:
    df0=next(pq.ParquetFile(fh).iter_batches(batch_size=500000,
            columns=['Zeit','Spannung','Strom','Zustand']).__iter__()).to_pandas()
df0['Zeit']=pd.to_datetime(df0['Zeit']); df0=df0.sort_values('Zeit').reset_index(drop=True)
all0=[]
for s,e in segments(df0,'CHA',1): all0.append(('CHA',s,e))
for s,e in segments(df0,'DCH',-1): all0.append(('DCH',s,e))
all0.sort(key=lambda x:x[1])
print(f"{'类型':>5}{'时长s':>8}{'V起':>8}{'V止':>8}{'V跨度':>8}{'容量Ah':>10}{'电流A':>8}")
for typ,s,e in all0[:6]:
    seg=df0.iloc[s:e]
    V=seg['Spannung'].values; I=seg['Strom'].values
    tt=(seg['Zeit']-seg['Zeit'].iloc[0]).dt.total_seconds().values
    Q=np.sum(np.abs(I)*np.diff(tt,prepend=tt[0])/3600)
    print(f"{typ:>5}{tt[-1]:>8.0f}{V[0]:>8.3f}{V[-1]:>8.3f}{abs(V[-1]-V[0]):>8.3f}{Q:>10.4f}{I.mean():>8.2f}")

=== 晚期文件: J8005_BMWK_METABatt=METABatt_A123_APR18650M1B_008=2026-01-27 ===
行数:500000, 状态分布:{'PAU': 163855, 'CHA': 132191, 'DCH': 132033, 'PAUO': 71916, 'REST': 5}

=== 晚期 充放电片段按时间顺序(看是否充放配对闭环) ===
   类型     时长s      V起      V止     V跨度      容量Ah     电流A
  DCH     995   3.200   1.999   1.201    0.1216   -0.44
  CHA    1064   2.765   3.600   0.835    0.7805    2.64
  DCH     720   3.297   3.133   0.164    0.5283   -2.64
  CHA     725   3.290   3.600   0.310    0.5317    2.64
  DCH     720   3.300   3.132   0.169    0.5283   -2.64
  CHA     725   3.290   3.600   0.311    0.5317    2.64
  DCH     720   3.300   3.132   0.168    0.5283   -2.64
  CHA     723   3.290   3.600   0.311    0.5305    2.64
  DCH     720   3.301   3.131   0.170    0.5282   -2.64
  CHA     722   3.289   3.600   0.311    0.5296    2.64
  DCH     720   3.301   3.131   0.170    0.5282   -2.64
  CHA     722   3.289   3.600   0.311    0.5293    2.64

画晚期一个充放循环...
图已存: 008_late_cycle.png

=== 早期对比(第一个文件) ===
   类型     时长s   

In [7]:
"""
单电池 DNN特征提取 —— 完整版(含3C自适应过滤)
进度对齐: 每个3C充放电片段在进度0→100%采16点
过滤: 只保留3C工况循环(电流匹配3C量级+时长够),排除标定/脉冲/完整放电
工况: aging文件名解析  |  SOH: 40_capacity_monitore
"""
import s3fs, re, numpy as np, pandas as pd
import pyarrow.parquet as pq
import warnings; warnings.filterwarnings('ignore')
from s3_credentials import S3_ACCESS_KEY, S3_SECRET_KEY

OPTS={"key":S3_ACCESS_KEY,"secret":S3_SECRET_KEY,
      "client_kwargs":{"endpoint_url":"https://iseadocker.isea.rwth-aachen.de:9000","region_name":"us-east-1"},
      "config_kwargs":{"s3":{"addressing_style":"path"},"signature_version":"s3v4"}}
RAWBASE="projects/j8005-metabatt/Metabatt/A123"
CAP="projects/j8005-metabatt/Metabatt/A123/40_capacity_monitore"
N_PTS=16
fs=s3fs.S3FileSystem(**OPTS)

BID="008"   # 改这个换电池

def parse_cond(aging_files):
    for f in aging_files:
        m=re.search(r'(\d+)grad_(\d+)SOC_(\d+)DOD_(\d+)C',f.split('/')[-1])
        if m: return {'Temp':int(m.group(1)),'SOC':int(m.group(2)),
                      'DOD':int(m.group(3)),'C_Rate':int(m.group(4))}
    return None

def load_soh(bid):
    f=f"{CAP}/METABatt_A123_APR18650M1B_{bid}_capacity.csv"
    if not fs.exists(f): return None,None,None
    c=pd.read_csv(f"s3://{f}",storage_options=OPTS).dropna(subset=['SOH'])
    if len(c)<2: return None,None,None
    c['t']=pd.to_datetime(c['CAP_start_time']); c=c.sort_values('t')
    return c['t'].astype('int64').values, c['SOH'].values, c['Capacity_py'].iloc[0]

def get_raw_segments(df,state,sign):
    m=((df['Zustand']==state)&(np.sign(df['Strom'])==sign)).values
    ch=np.diff(m.astype(int)); st=np.where(ch==1)[0]+1; en=np.where(ch==-1)[0]+1
    if len(m)>0 and m[0]: st=np.r_[0,st]
    if len(en)<len(st): en=np.r_[en,len(df)]
    return [df.iloc[s:e] for s,e in zip(st,en) if e-s>=30]

def is_3c_cycle(seg, C0):
    """只保留3C工况循环: 电流匹配3C量级(随老化在范围内) + 时长够(排脉冲)"""
    I=abs(seg['Strom'].mean())
    tt=(pd.to_datetime(seg['Zeit'])-pd.to_datetime(seg['Zeit']).iloc[0]).dt.total_seconds().values
    dur=tt[-1]
    I_lo = 2.0 * C0 * 0.7    # 3C下限(容量衰减到~0.7×C0时的3C)
    I_hi = 3.5 * C0          # 3C上限(留余量)
    if not (I_lo < I < I_hi): return False   # 电流不是3C→排除(标定<1C,在此之下)
    if dur < 60: return False                 # 太短→脉冲,排除
    return True

def extract_curves(seg):
    """进度对齐: 电压曲线16 + IC曲线16"""
    V=seg['Spannung'].values; I=seg['Strom'].values
    tt=(pd.to_datetime(seg['Zeit'])-pd.to_datetime(seg['Zeit']).iloc[0]).dt.total_seconds().values
    if len(V)<30 or tt[-1]<=0: return None,None
    Q=np.cumsum(np.abs(I)*np.diff(tt,prepend=tt[0])/3600.0)
    if Q[-1]<=0: return None,None
    prog=Q/Q[-1]; grid=np.linspace(0,1,N_PTS)
    Vg=np.interp(grid,prog,V)
    Qg=np.interp(grid,prog,Q)
    dV=np.gradient(Vg); dQ=np.gradient(Qg)
    with np.errstate(divide='ignore',invalid='ignore'):
        IC=np.where(np.abs(dV)>1e-6, dQ/dV, 0.0)
    IC=np.nan_to_num(IC,nan=0.0,posinf=0.0,neginf=0.0)
    return Vg, IC

# ================= 主流程 =================
BASE=f"{RAWBASE}/METABatt_A123_APR18650M1B_{BID}"
files=fs.ls(BASE)
aging=sorted([f for f in files if 'Aging' in f and 'Cyc' in f])
assert aging, f"{BID} 无aging"
cond=parse_cond(aging); assert cond, "工况解析失败"
assert cond['C_Rate']==3, f"{BID}非3C"
soh_t,soh_v,C0=load_soh(BID); assert soh_t is not None, "无SOH"

print(f"电池{BID}: {cond['Temp']}℃/{cond['SOC']}SOC/{cond['DOD']}DOD/{cond['C_Rate']}C")
print(f"C0={C0:.4f}Ah, 3C电流范围≈{2.0*C0*0.7:.2f}~{3.5*C0:.2f}A, SOH {soh_v.max():.1f}→{soh_v.min():.1f}")

rows=[]; n_raw_dch=0; n_kept_dch=0
for fi,f in enumerate(aging):
    with fs.open(f) as fh:
        df=next(pq.ParquetFile(fh).iter_batches(batch_size=500000,
                columns=['Zeit','Spannung','Strom','Zustand']).__iter__()).to_pandas()
    df['Zeit']=pd.to_datetime(df['Zeit'])
    dch=get_raw_segments(df,'DCH',-1); cha=get_raw_segments(df,'CHA',1)
    n_raw_dch+=len(dch)
    # 过滤:只留3C循环
    dch=[s for s in dch if is_3c_cycle(s,C0)]
    cha=[s for s in cha if is_3c_cycle(s,C0)]
    n_kept_dch+=len(dch)
    for i in range(min(len(dch),len(cha))):
        Vdis,ICdis=extract_curves(dch[i])
        Vchg,_=extract_curves(cha[i])
        if Vdis is None or Vchg is None: continue
        t=pd.to_datetime(dch[i]['Zeit'].iloc[0])
        soh=float(np.interp(t.value,soh_t,soh_v))
        row={'bid':BID,'SOC':cond['SOC'],'DOD':cond['DOD'],'Temp':cond['Temp'],
             'C_Rate':cond['C_Rate'],'SOH':soh,'time':t}
        for k in range(N_PTS): row[f'dis_{k}']=Vdis[k]
        for k in range(N_PTS): row[f'chg_{k}']=Vchg[k]
        for k in range(N_PTS): row[f'ic_{k}']=ICdis[k]
        rows.append(row)
    print(f"  [{fi+1}/{len(aging)}] 原始DCH{len(get_raw_segments(df,'DCH',-1))} 过滤后{len(dch)} 累计{len(rows)}")

data=pd.DataFrame(rows)
data.to_csv(f"{BID}_dnn_samples.csv",index=False)
print(f"\n✅ 提取{len(data)}样本 (原始DCH段{n_raw_dch}→3C循环{n_kept_dch})")
print(f"SOH范围: {data['SOH'].min():.1f}~{data['SOH'].max():.1f}")

# 自检:第一个样本(应该是干净的3C循环,放电跨度~0.15V不到2.0V)
r0=data.iloc[0]
print(f"\n第一个样本放电曲线: {[round(r0[f'dis_{k}'],3) for k in range(N_PTS)]}")
print(f"  → 末点{r0['dis_15']:.3f}V (应该~3.0-3.15,不是2.0!跨度{r0['dis_0']-r0['dis_15']:.3f})")

# 早期相关性
print(f"\n=== 放电曲线各点与SOH相关性(全程/早期SOH>92) ===")
es=data[data['SOH']>92]
for k in [0,4,8,12,15]:
    rg=data[f'dis_{k}'].corr(data['SOH'])
    re_=es[f'dis_{k}'].corr(es['SOH']) if len(es)>10 else float('nan')
    print(f"  dis_{k}(进度{int(k/15*100)}%): 全程{rg:+.3f} 早期{re_:+.3f}")
print(f"\nDNN输入: X_curves[{len(data)},16,3] X_cond[{len(data)},4] Y[{len(data)}]")

电池008: 35℃/70SOC/60DOD/3C
C0=1.1585Ah, 3C电流范围≈1.62~4.05A, SOH 96.5→72.4
  [1/21] 原始DCH87 过滤后86 累计86
  [2/21] 原始DCH50 过滤后49 累计135
  [3/21] 原始DCH92 过滤后91 累计226
  [4/21] 原始DCH98 过滤后97 累计323
  [5/21] 原始DCH97 过滤后96 累计419
  [6/21] 原始DCH98 过滤后97 累计516
  [7/21] 原始DCH97 过滤后96 累计612
  [8/21] 原始DCH99 过滤后97 累计709
  [9/21] 原始DCH98 过滤后97 累计806
  [10/21] 原始DCH76 过滤后75 累计881
  [11/21] 原始DCH84 过滤后83 累计964
  [12/21] 原始DCH81 过滤后80 累计1044
  [13/21] 原始DCH98 过滤后97 累计1141
  [14/21] 原始DCH98 过滤后97 累计1238
  [15/21] 原始DCH98 过滤后97 累计1335
  [16/21] 原始DCH100 过滤后97 累计1432
  [17/21] 原始DCH93 过滤后92 累计1524
  [18/21] 原始DCH96 过滤后95 累计1619
  [19/21] 原始DCH97 过滤后96 累计1715
  [20/21] 原始DCH97 过滤后96 累计1811
  [21/21] 原始DCH90 过滤后89 累计1900

✅ 提取1900样本 (原始DCH段1924→3C循环1900)
SOH范围: 73.0~96.5

第一个样本放电曲线: [np.float64(3.289), np.float64(3.237), np.float64(3.231), np.float64(3.225), np.float64(3.22), np.float64(3.215), np.float64(3.21), np.float64(3.205), np.float64(3.199), np.float64(3.193), np.float64(3.187), np.float64(3.18), np.float

In [9]:
"""
模块1: 找可用电池
条件: 有aging文件 + 文件名是3C + 40_capacity_monitore里有SOH
输出: 可用电池清单 usable_batteries.csv
"""
import s3fs, re, pandas as pd
from s3_credentials import S3_ACCESS_KEY, S3_SECRET_KEY
OPTS={"key":S3_ACCESS_KEY,"secret":S3_SECRET_KEY,
      "client_kwargs":{"endpoint_url":"https://iseadocker.isea.rwth-aachen.de:9000","region_name":"us-east-1"},
      "config_kwargs":{"s3":{"addressing_style":"path"},"signature_version":"s3v4"}}
RAWBASE="projects/j8005-metabatt/Metabatt/A123"
CAP="projects/j8005-metabatt/Metabatt/A123/40_capacity_monitore"
fs=s3fs.S3FileSystem(**OPTS)

def parse_cond(af):
    for f in af:
        m=re.search(r'(\d+)grad_(\d+)SOC_(\d+)DOD_(\d+)C',f.split('/')[-1])
        if m: return {'Temp':int(m.group(1)),'SOC':int(m.group(2)),
                      'DOD':int(m.group(3)),'C_Rate':int(m.group(4))}
    return None

# 列出所有电池文件夹
all_dirs=fs.ls(RAWBASE)
all_bids=sorted([re.search(r'_(\d+)$',d).group(1) for d in all_dirs
                 if re.search(r'METABatt_A123_APR18650M1B_(\d+)$',d)],key=int)
print(f"共{len(all_bids)}个电池文件夹,开始筛选...\n")

usable=[]; skip_reasons={}
for bid in all_bids:
    BASE=f"{RAWBASE}/METABatt_A123_APR18650M1B_{bid}"
    try: files=fs.ls(BASE)
    except: skip_reasons[bid]="无法访问"; continue
    aging=[f for f in files if 'Aging' in f and 'Cyc' in f]
    if not aging: skip_reasons[bid]="无aging"; continue
    cond=parse_cond(aging)
    if not cond: skip_reasons[bid]="工况解析失败"; continue
    if cond['C_Rate']!=3: skip_reasons[bid]=f"非3C({cond['C_Rate']}C)"; continue
    # 检查capacity文件
    capf=f"{CAP}/METABatt_A123_APR18650M1B_{bid}_capacity.csv"
    if not fs.exists(capf): skip_reasons[bid]="无capacity"; continue
    try:
        c=pd.read_csv(f"s3://{capf}",storage_options=OPTS).dropna(subset=['SOH'])
        if len(c)<2: skip_reasons[bid]="SOH点不足"; continue
        soh_max,soh_min=c['SOH'].max(),c['SOH'].min()
    except: skip_reasons[bid]="capacity读取失败"; continue
    usable.append({'bid':bid,'Temp':cond['Temp'],'SOC':cond['SOC'],'DOD':cond['DOD'],
                   'C_Rate':cond['C_Rate'],'n_aging':len(aging),
                   'n_checkup':len(c),'SOH_max':round(soh_max,1),'SOH_min':round(soh_min,1)})
    print(f"  ✓ {bid}: {cond['Temp']}℃/{cond['SOC']}SOC/{cond['DOD']}DOD, "
          f"{len(aging)}aging, SOH{soh_max:.0f}→{soh_min:.0f}")

ud=pd.DataFrame(usable)
ud.to_csv("usable_batteries.csv",index=False)
print(f"\n========== 筛选结果 ==========")
print(f"可用电池: {len(ud)} 块")
print(f"\n跳过原因统计:")
from collections import Counter
for reason,cnt in Counter(skip_reasons.values()).items():
    print(f"  {reason}: {cnt}块")

print(f"\n=== 可用电池工况分布 ===")
print(f"温度: {dict(ud.groupby('Temp').size())}")
print(f"SOC×DOD分布:")
print(pd.crosstab(ud['SOC'],ud['DOD']))
print(f"\n总预计样本: 约{ud['n_aging'].sum()*90:.0f} (按每aging文件~90循环估)")

共423个电池文件夹,开始筛选...

  ✓ 007: 35℃/70SOC/60DOD, 22aging, SOH100→72
  ✓ 008: 35℃/70SOC/60DOD, 21aging, SOH96→72
  ✓ 009: 35℃/70SOC/60DOD, 22aging, SOH97→72
  ✓ 010: 35℃/70SOC/60DOD, 22aging, SOH97→72
  ✓ 011: 35℃/30SOC/60DOD, 21aging, SOH100→80
  ✓ 012: 35℃/70SOC/20DOD, 22aging, SOH101→90
  ✓ 013: 35℃/70SOC/20DOD, 22aging, SOH101→89
  ✓ 014: 35℃/70SOC/20DOD, 22aging, SOH101→87
  ✓ 015: 35℃/70SOC/20DOD, 22aging, SOH102→88
  ✓ 016: 35℃/70SOC/60DOD, 22aging, SOH97→72
  ✓ 017: 35℃/70SOC/60DOD, 22aging, SOH97→72
  ✓ 018: 35℃/70SOC/60DOD, 22aging, SOH96→73
  ✓ 019: 35℃/70SOC/60DOD, 22aging, SOH97→72
  ✓ 020: 35℃/50SOC/60DOD, 22aging, SOH100→81
  ✓ 021: 35℃/30SOC/60DOD, 22aging, SOH101→83
  ✓ 022: 35℃/30SOC/60DOD, 22aging, SOH101→78
  ✓ 023: 35℃/30SOC/60DOD, 22aging, SOH101→84
  ✓ 024: 35℃/30SOC/60DOD, 22aging, SOH100→79
  ✓ 025: 35℃/30SOC/60DOD, 21aging, SOH102→84
  ✓ 027: 35℃/30SOC/60DOD, 22aging, SOH100→79
  ✓ 028: 35℃/50SOC/100DOD, 18aging, SOH100→63
  ✓ 032: 35℃/30SOC/60DOD, 22aging, SOH101

In [3]:
"""
IC曲线可视化 —— 看峰在哪、和其他地方的对比、随老化变化
画早/中/晚期循环的IC曲线,标出峰
"""
import s3fs, re, numpy as np, pandas as pd
import pyarrow.parquet as pq
from scipy.ndimage import gaussian_filter1d
from scipy.signal import find_peaks
import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings('ignore')
from s3_credentials import S3_ACCESS_KEY, S3_SECRET_KEY
OPTS={"key":S3_ACCESS_KEY,"secret":S3_SECRET_KEY,
      "client_kwargs":{"endpoint_url":"https://iseadocker.isea.rwth-aachen.de:9000","region_name":"us-east-1"},
      "config_kwargs":{"s3":{"addressing_style":"path"},"signature_version":"s3v4"}}
RAWBASE="projects/j8005-metabatt/Metabatt/A123"
CAP="projects/j8005-metabatt/Metabatt/A123/40_capacity_monitore"
fs=s3fs.S3FileSystem(**OPTS)
BID="008"; DV=0.005; SIGMA=2; PROM=0.15; TOL=0.25

def build_cap_interp(bid):
    c=pd.read_csv(f"s3://{CAP}/METABatt_A123_APR18650M1B_{bid}_capacity.csv",
                  storage_options=OPTS).dropna(subset=['Capacity_py'])
    c['t']=pd.to_datetime(c['CAP_start_time']); c=c.sort_values('t')
    return c['t'].astype('int64').values.astype(float), c['Capacity_py'].values
def get_raw(df,state,sign):
    m=((df['Zustand']==state)&(np.sign(df['Strom'])==sign)).values
    ch=np.diff(m.astype(int));st=np.where(ch==1)[0]+1;en=np.where(ch==-1)[0]+1
    if len(m)>0 and m[0]:st=np.r_[0,st]
    if len(en)<len(st):en=np.r_[en,len(df)]
    return [df.iloc[s:e] for s,e in zip(st,en) if e-s>=30]
def is_3c(seg,cap_t,cap_v):
    t=pd.to_datetime(seg['Zeit'].iloc[0]).value
    C_now=np.interp(t,cap_t,cap_v); It=3*C_now
    I=abs(seg['Strom'].mean())
    tt=(pd.to_datetime(seg['Zeit'])-pd.to_datetime(seg['Zeit']).iloc[0]).dt.total_seconds().values
    return (It*(1-TOL)<I<It*(1+TOL)) and tt[-1]>=60

def compute_ic(seg):
    """返回 vgrid, IC, 峰位, 峰高 + 中间量(便于排查)"""
    V=seg['Spannung'].values; I=seg['Strom'].values
    tt=(pd.to_datetime(seg['Zeit'])-pd.to_datetime(seg['Zeit']).iloc[0]).dt.total_seconds().values
    Q=np.cumsum(np.abs(I)*np.diff(tt,prepend=tt[0])/3600.0)
    info={'n_points':len(V),'Q_total':Q[-1] if len(Q) else 0,
          'V_range':(V.min(),V.max()) if len(V) else (0,0)}
    if Q[-1]<=0: return None,info,"Q_total<=0"
    o=np.argsort(V); Vu,idx=np.unique(V[o],return_index=True); Qu=Q[o][idx]
    info['n_unique_V']=len(Vu)
    if len(Vu)<10: return None,info,"unique电压点<10"
    vgrid=np.arange(Vu.min(),Vu.max(),DV)
    info['n_vgrid']=len(vgrid)
    if len(vgrid)<10: return None,info,"vgrid点<10(电压跨度太小)"
    Qsm=gaussian_filter1d(np.interp(vgrid,Vu,Qu),sigma=SIGMA)
    IC=np.abs(np.gradient(Qsm,vgrid))
    if IC.max()<=0: return None,info,"IC全0"
    peaks,props=find_peaks(IC,prominence=IC.max()*PROM)
    info['n_peaks']=len(peaks)
    if len(peaks)==0:
        return {'vgrid':vgrid,'IC':IC,'Vpeak':None,'ICpeak':None},info,"未找到峰(IC单调?)"
    main=peaks[np.argmax(IC[peaks])]
    return {'vgrid':vgrid,'IC':IC,'Vpeak':vgrid[main],'ICpeak':IC[main],
            'all_peaks':vgrid[peaks]},info,"OK"

# ===== 取早/中/晚期循环 =====
BASE=f"{RAWBASE}/METABatt_A123_APR18650M1B_{BID}"
aging=sorted([f for f in fs.ls(BASE) if 'Aging' in f and 'Cyc' in f])
cap_t,cap_v=build_cap_interp(BID)
print(f"电池{BID}: {len(aging)}个aging文件\n")

picks=[('早期',0),('中期',len(aging)//2),('晚期',len(aging)-1)]
fig,axes=plt.subplots(1,3,figsize=(18,5))
colors={'早期':'b','中期':'g','晚期':'r'}

for ax,(label,fi) in zip(axes,picks):
    with fs.open(aging[fi]) as fh:
        df=next(pq.ParquetFile(fh).iter_batches(batch_size=500000,
                columns=['Zeit','Spannung','Strom','Zustand']).__iter__()).to_pandas()
    df['Zeit']=pd.to_datetime(df['Zeit'])
    # 找第一个3C放电
    seg=None
    for s in get_raw(df,'DCH',-1):
        if is_3c(s,cap_t,cap_v): seg=s; break
    if seg is None:
        print(f"{label}: 无3C放电段"); ax.set_title(f"{label}: 无3C段"); continue
    res,info,status=compute_ic(seg)
    print(f"=== {label}(文件{fi}) ===")
    print(f"  原始点数{info['n_points']}, Q总{info.get('Q_total',0):.3f}Ah, "
          f"电压范围{info['V_range'][0]:.3f}~{info['V_range'][1]:.3f}V")
    print(f"  vgrid点{info.get('n_vgrid','?')}, 找到峰数{info.get('n_peaks','?')}, 状态:{status}")
    if res is None:
        print(f"  ✗ 无法算IC: {status}"); ax.set_title(f"{label}: {status}"); continue
    # 画IC曲线
    ax.plot(res['vgrid'],res['IC'],color=colors[label],lw=1.5,label='IC=|dQ/dV|')
    if res['Vpeak'] is not None:
        print(f"  ✓ 主峰位={res['Vpeak']:.4f}V, 峰高={res['ICpeak']:.2f}")
        # 标主峰
        ax.axvline(res['Vpeak'],color='k',ls='--',alpha=0.7)
        ax.plot(res['Vpeak'],res['ICpeak'],'k*',ms=18,label=f"主峰{res['Vpeak']:.3f}V")
        # 标所有峰
        for vp in res['all_peaks']:
            ax.plot(vp,res['IC'][np.argmin(np.abs(res['vgrid']-vp))],'o',
                    color='orange',ms=8,alpha=0.6)
    ax.set_title(f"{label}(文件{fi})"); ax.set_xlabel('电压V'); ax.set_ylabel('IC=|dQ/dV|')
    ax.legend(); ax.grid(alpha=0.3)

plt.suptitle(f'电池{BID} IC曲线与峰(早/中/晚期) —— 星=主峰,橙点=其他峰',fontsize=13)
plt.tight_layout(); plt.savefig(f'{BID}_IC_peaks.png',dpi=110)
print(f"\n图已存: {BID}_IC_peaks.png")
print("看图:主峰(黑星)应明显高于周围;早中晚峰位/峰高的变化=老化")

电池008: 21个aging文件

=== 早期(文件0) ===
  原始点数1455, Q总0.712Ah, 电压范围3.152~3.289V
  vgrid点28, 找到峰数1, 状态:OK
  ✓ 主峰位=3.2121V, 峰高=8.95
=== 中期(文件10) ===
  原始点数1458, Q总0.586Ah, 电压范围3.148~3.299V
  vgrid点31, 找到峰数1, 状态:OK
  ✓ 主峰位=3.2128V, 峰高=7.53
=== 晚期(文件20) ===
  原始点数1459, Q总0.528Ah, 电压范围3.133~3.297V
  vgrid点33, 找到峰数1, 状态:OK
  ✓ 主峰位=3.2079V, 峰高=6.13

图已存: 008_IC_peaks.png
看图:主峰(黑星)应明显高于周围;早中晚峰位/峰高的变化=老化


In [22]:
"""
单块验证 峰锚定提取 —— 无SOC/DOD版
只跑一块,验证:锚定率、Vpeak0、dVpeak随老化、特征与SOH相关性
输出不含SOC/DOD,只有Temp
"""
import s3fs, re, numpy as np, pandas as pd
import pyarrow.parquet as pq
from scipy.ndimage import gaussian_filter1d
from scipy.signal import find_peaks
from scipy.interpolate import PchipInterpolator
import warnings; warnings.filterwarnings('ignore')
from s3_credentials import S3_ACCESS_KEY, S3_SECRET_KEY

OPTS={"key":S3_ACCESS_KEY,"secret":S3_SECRET_KEY,
      "client_kwargs":{"endpoint_url":"https://iseadocker.isea.rwth-aachen.de:9000","region_name":"us-east-1"},
      "config_kwargs":{"s3":{"addressing_style":"path"},"signature_version":"s3v4"}}
RAWBASE="projects/j8005-metabatt/Metabatt/A123"
CAP="projects/j8005-metabatt/Metabatt/A123/40_capacity_monitore"
fs=s3fs.S3FileSystem(**OPTS)

BID="013"
N_PTS=16; HALF_WIN=0.06; DV=0.005; SIGMA=2; TOL=0.25; PROM=0.15

def parse_temp_crate(af):
    for f in af:
        m=re.search(r'(\d+)grad_(\d+)SOC_(\d+)DOD_(\d+)C',f.split('/')[-1])
        if m: return {'Temp':int(m.group(1)),'C_Rate':int(m.group(4))}
    return None
def build_pchip_soh(bid):
    c=pd.read_csv(f"s3://{CAP}/METABatt_A123_APR18650M1B_{bid}_capacity.csv",
                  storage_options=OPTS).dropna(subset=['SOH'])
    c['t']=pd.to_datetime(c['CAP_start_time']); c=c.sort_values('t').reset_index(drop=True)
    t_ns=c['t'].astype('int64').values.astype(float); soh=c['SOH'].values.copy()
    for i in range(1,len(soh)):
        if soh[i]>soh[i-1]: soh[i]=soh[i-1]
    return PchipInterpolator(t_ns,soh,extrapolate=True), c['Capacity_py'].iloc[0]
def build_cap_interp(bid):
    c=pd.read_csv(f"s3://{CAP}/METABatt_A123_APR18650M1B_{bid}_capacity.csv",
                  storage_options=OPTS).dropna(subset=['Capacity_py'])
    c['t']=pd.to_datetime(c['CAP_start_time']); c=c.sort_values('t')
    return c['t'].astype('int64').values.astype(float), c['Capacity_py'].values
def get_raw(df,state,sign):
    m=((df['Zustand']==state)&(np.sign(df['Strom'])==sign)).values
    ch=np.diff(m.astype(int));st=np.where(ch==1)[0]+1;en=np.where(ch==-1)[0]+1
    if len(m)>0 and m[0]:st=np.r_[0,st]
    if len(en)<len(st):en=np.r_[en,len(df)]
    return [df.iloc[s:e] for s,e in zip(st,en) if e-s>=30]
def is_3c_dynamic(seg,cap_t,cap_v):
    t=pd.to_datetime(seg['Zeit'].iloc[0]).value
    C_now=np.interp(t,cap_t,cap_v); It=3*C_now
    I=abs(seg['Strom'].mean())
    tt=(pd.to_datetime(seg['Zeit'])-pd.to_datetime(seg['Zeit']).iloc[0]).dt.total_seconds().values
    return (It*(1-TOL)<I<It*(1+TOL)) and tt[-1]>=60
def get_ic_peak(seg):
    V=seg['Spannung'].values; I=seg['Strom'].values
    tt=(pd.to_datetime(seg['Zeit'])-pd.to_datetime(seg['Zeit']).iloc[0]).dt.total_seconds().values
    Q=np.cumsum(np.abs(I)*np.diff(tt,prepend=tt[0])/3600.0)
    if Q[-1]<=0: return None
    o=np.argsort(V); Vu,idx=np.unique(V[o],return_index=True); Qu=Q[o][idx]
    if len(Vu)<10: return None
    vgrid=np.arange(Vu.min(),Vu.max(),DV)
    if len(vgrid)<2*N_PTS: return None
    Qsm=gaussian_filter1d(np.interp(vgrid,Vu,Qu),sigma=SIGMA)
    IC=np.abs(np.gradient(Qsm,vgrid))
    if IC.max()<=0: return None
    peaks,_=find_peaks(IC,prominence=IC.max()*PROM)
    if len(peaks)==0: return None
    Vpeak=vgrid[peaks[np.argmax(IC[peaks])]]
    return vgrid,Qsm,IC,Vpeak
def reconstruct(seg,ref_refQ,ref_dvaxis,Vpeak0):
    r=get_ic_peak(seg)
    if r is None: return None
    vgrid,Qsm,IC,Vpeak=r
    dv=np.linspace(-HALF_WIN,HALF_WIN,N_PTS); absv=Vpeak+dv
    Q16=np.interp(absv,vgrid,Qsm,left=np.nan,right=np.nan)
    IC16=np.interp(absv,vgrid,IC,left=np.nan,right=np.nan)
    if np.isnan(Q16).any(): return None
    dQ16=Q16-np.interp(dv,ref_dvaxis,ref_refQ) if ref_refQ is not None else np.zeros(N_PTS)
    return {'Q':Q16,'dQ':dQ16,'IC':IC16,'Vpeak':Vpeak,'dVpeak':Vpeak0-Vpeak}

# ===== 主流程 =====
BASE=f"{RAWBASE}/METABatt_A123_APR18650M1B_{BID}"
aging=sorted([f for f in fs.ls(BASE) if 'Aging' in f and 'Cyc' in f])
tc=parse_temp_crate(aging)
pchip,C0=build_pchip_soh(BID); cap_t,cap_v=build_cap_interp(BID)
print(f"电池{BID}: Temp={tc['Temp']}℃, C0={C0:.4f} (不读取SOC/DOD)")

# 第一遍:Vpeak0
Vpeak0=None; ref_refQ=None; ref_dvaxis=np.linspace(-HALF_WIN,HALF_WIN,N_PTS)
with fs.open(aging[0]) as fh:
    df0=next(pq.ParquetFile(fh).iter_batches(batch_size=500000,
            columns=['Zeit','Spannung','Strom','Zustand']).__iter__()).to_pandas()
df0['Zeit']=pd.to_datetime(df0['Zeit'])
for seg in get_raw(df0,'DCH',-1):
    if not is_3c_dynamic(seg,cap_t,cap_v): continue
    r=get_ic_peak(seg)
    if r is None: continue
    vgrid,Qsm,IC,Vpeak=r
    refQ=np.interp(Vpeak+ref_dvaxis,vgrid,Qsm,left=np.nan,right=np.nan)
    if not np.isnan(refQ).any():
        Vpeak0=Vpeak; ref_refQ=refQ; break
print(f"参考峰位 Vpeak0={Vpeak0:.4f}V")

# 第二遍
rows=[]; n_total=0; n_anchor=0
for fi,f in enumerate(aging):
    with fs.open(f) as fh:
        df=next(pq.ParquetFile(fh).iter_batches(batch_size=500000,
                columns=['Zeit','Spannung','Strom','Zustand']).__iter__()).to_pandas()
    df['Zeit']=pd.to_datetime(df['Zeit'])
    for seg in get_raw(df,'DCH',-1):
        if not is_3c_dynamic(seg,cap_t,cap_v): continue
        n_total+=1
        feat=reconstruct(seg,ref_refQ,ref_dvaxis,Vpeak0)
        if feat is None: continue
        n_anchor+=1
        t=pd.to_datetime(seg['Zeit'].iloc[0])
        soh=float(pchip(t.value))
        row={'bid':BID,'Temp':tc['Temp'],'SOH':soh,'time':t,
             'Vpeak':feat['Vpeak'],'Vpeak0':Vpeak0,'dVpeak':feat['dVpeak']}
        for k in range(N_PTS): row[f'Q_{k}']=feat['Q'][k]
        for k in range(N_PTS): row[f'dQ_{k}']=feat['dQ'][k]
        for k in range(N_PTS): row[f'IC_{k}']=feat['IC'][k]
        rows.append(row)
    print(f"  [{fi+1}/{len(aging)}] 累计样本{len(rows)}")

data=pd.DataFrame(rows)
data.to_csv(f"{BID}_peak_features.csv",index=False)
print(f"\n✅ {len(data)}样本 → {BID}_peak_features.csv")
print(f"锚定成功率: {n_anchor}/{n_total} = {n_anchor/max(n_total,1)*100:.1f}%")
print(f"列: {[c for c in data.columns if not c.startswith(('Q_','dQ_','IC_'))]} + Q/dQ/IC各16 (无SOC/DOD)")

# 验证
d=data.sort_values('SOH',ascending=False)
print(f"\n=== 验证 ===")
print(f"Vpeak随老化: 早期{d['Vpeak'].head(50).mean():.4f} → 晚期{d['Vpeak'].tail(50).mean():.4f}V "
      f"(横移{(d['Vpeak'].head(50).mean()-d['Vpeak'].tail(50).mean())*1000:+.1f}mV)")
print(f"dVpeak随老化: 早期{d['dVpeak'].head(50).mean():+.4f} → 晚期{d['dVpeak'].tail(50).mean():+.4f}")
print(f"\n特征与SOH相关性(全程/早期SOH>92):")
es=data[data['SOH']>92]
for col in ['dVpeak','Vpeak','IC_8','Q_8','dQ_8']:
    rg=data[col].corr(data['SOH'])
    re_=es[col].corr(es['SOH']) if len(es)>10 else float('nan')
    print(f"  {col:8s}: 全程{rg:+.3f} 早期{re_:+.3f}")


ValueError: time data "2025-10-15 15:58:28+00:00" doesn't match format "%Y-%m-%d %H:%M:%S.%f%z", at position 14. You might want to try:
    - passing `format` if your strings have a consistent format;
    - passing `format='ISO8601'` if your strings are all ISO8601 but not necessarily in exactly the same format;
    - passing `format='mixed'`, and the format will be inferred for each element individually. You might want to use `dayfirst` alongside this.

In [ ]:
"""
5电池验证 —— HALF_WIN=0.06(改回) + DV=0.002(加密网格)
救回早期窄跨度循环,看锚定率和IC特征
"""
import s3fs, re, numpy as np, pandas as pd
import pyarrow.parquet as pq
from scipy.ndimage import gaussian_filter1d
from scipy.signal import find_peaks
from scipy.interpolate import PchipInterpolator
from collections import Counter
import warnings; warnings.filterwarnings('ignore')
from s3_credentials import S3_ACCESS_KEY, S3_SECRET_KEY

OPTS={"key":S3_ACCESS_KEY,"secret":S3_SECRET_KEY,
      "client_kwargs":{"endpoint_url":"https://iseadocker.isea.rwth-aachen.de:9000","region_name":"us-east-1"},
      "config_kwargs":{"s3":{"addressing_style":"path"},"signature_version":"s3v4"}}
RAWBASE="projects/j8005-metabatt/Metabatt/A123"
CAP="projects/j8005-metabatt/Metabatt/A123/40_capacity_monitore"
fs=s3fs.S3FileSystem(**OPTS)

N_PTS=16; HALF_WIN=0.06; DV=0.002; SIGMA=2; TOL=0.25; PROM=0.15   # 窗口0.06, 网格0.002

# 5块不同工况验证
VERIFY=['008','011','012','028','125']   # 60DOD/30SOC60DOD/20DOD/100DOD/10SOC20DOD(浅)

def parse_temp_crate(af):
    for f in af:
        m=re.search(r'(\d+)grad_(\d+)SOC_(\d+)DOD_(\d+)C',f.split('/')[-1])
        if m: return {'Temp':int(m.group(1)),'C_Rate':int(m.group(4))}
    return None
def build_pchip_soh(bid):
    c=pd.read_csv(f"s3://{CAP}/METABatt_A123_APR18650M1B_{bid}_capacity.csv",
                  storage_options=OPTS).dropna(subset=['SOH'])
    c['t']=pd.to_datetime(c['CAP_start_time']); c=c.sort_values('t').reset_index(drop=True)
    t_ns=c['t'].astype('int64').values.astype(float); soh=c['SOH'].values.copy()
    for i in range(1,len(soh)):
        if soh[i]>soh[i-1]: soh[i]=soh[i-1]
    return PchipInterpolator(t_ns,soh,extrapolate=True), c['Capacity_py'].iloc[0]
def build_cap_interp(bid):
    c=pd.read_csv(f"s3://{CAP}/METABatt_A123_APR18650M1B_{bid}_capacity.csv",
                  storage_options=OPTS).dropna(subset=['Capacity_py'])
    c['t']=pd.to_datetime(c['CAP_start_time']); c=c.sort_values('t')
    return c['t'].astype('int64').values.astype(float), c['Capacity_py'].values
def get_raw(df,state,sign):
    m=((df['Zustand']==state)&(np.sign(df['Strom'])==sign)).values
    ch=np.diff(m.astype(int));st=np.where(ch==1)[0]+1;en=np.where(ch==-1)[0]+1
    if len(m)>0 and m[0]:st=np.r_[0,st]
    if len(en)<len(st):en=np.r_[en,len(df)]
    return [df.iloc[s:e] for s,e in zip(st,en) if e-s>=30]
def is_3c_dynamic(seg,cap_t,cap_v):
    t=pd.to_datetime(seg['Zeit'].iloc[0]).value
    C_now=np.interp(t,cap_t,cap_v); It=3*C_now
    I=abs(seg['Strom'].mean())
    tt=(pd.to_datetime(seg['Zeit'])-pd.to_datetime(seg['Zeit']).iloc[0]).dt.total_seconds().values
    return (It*(1-TOL)<I<It*(1+TOL)) and tt[-1]>=60
def get_ic_peak(seg):
    V=seg['Spannung'].values; I=seg['Strom'].values
    tt=(pd.to_datetime(seg['Zeit'])-pd.to_datetime(seg['Zeit']).iloc[0]).dt.total_seconds().values
    Q=np.cumsum(np.abs(I)*np.diff(tt,prepend=tt[0])/3600.0)
    if Q[-1]<=0: return None,"Q<=0"
    o=np.argsort(V); Vu,idx=np.unique(V[o],return_index=True); Qu=Q[o][idx]
    if len(Vu)<10: return None,"unique<10"
    vgrid=np.arange(Vu.min(),Vu.max(),DV)
    if len(vgrid)<2*N_PTS: return None,"vgrid不足"
    Qsm=gaussian_filter1d(np.interp(vgrid,Vu,Qu),sigma=SIGMA)
    IC=np.abs(np.gradient(Qsm,vgrid))
    if IC.max()<=0: return None,"IC全0"
    peaks,_=find_peaks(IC,prominence=IC.max()*PROM)
    if len(peaks)==0: return None,"无峰"
    Vpeak=vgrid[peaks[np.argmax(IC[peaks])]]
    return (vgrid,Qsm,IC,Vpeak),"OK"
def reconstruct(seg,ref_refQ,ref_dvaxis,Vpeak0):
    r,why=get_ic_peak(seg)
    if r is None: return None,why
    vgrid,Qsm,IC,Vpeak=r
    dv=np.linspace(-HALF_WIN,HALF_WIN,N_PTS); absv=Vpeak+dv
    Q16=np.interp(absv,vgrid,Qsm,left=np.nan,right=np.nan)
    IC16=np.interp(absv,vgrid,IC,left=np.nan,right=np.nan)
    if np.isnan(Q16).any(): return None,"窗口超界"
    dQ16=Q16-np.interp(dv,ref_dvaxis,ref_refQ)
    return {'Q':Q16,'dQ':dQ16,'IC':IC16,'Vpeak':Vpeak,'dVpeak':Vpeak0-Vpeak},"OK"

def process(bid):
    BASE=f"{RAWBASE}/METABatt_A123_APR18650M1B_{bid}"
    aging=sorted([f for f in fs.ls(BASE) if 'Aging' in f and 'Cyc' in f])
    tc=parse_temp_crate(aging); pchip,C0=build_pchip_soh(bid); cap_t,cap_v=build_cap_interp(bid)
    ref_dvaxis=np.linspace(-HALF_WIN,HALF_WIN,N_PTS); Vpeak0=None; ref_refQ=None
    with fs.open(aging[0]) as fh:
        df0=next(pq.ParquetFile(fh).iter_batches(batch_size=500000,
                columns=['Zeit','Spannung','Strom','Zustand']).__iter__()).to_pandas()
    df0['Zeit']=pd.to_datetime(df0['Zeit'])
    for seg in get_raw(df0,'DCH',-1):
        if not is_3c_dynamic(seg,cap_t,cap_v): continue
        r,_=get_ic_peak(seg)
        if r is None: continue
        vgrid,Qsm,IC,Vpeak=r
        refQ=np.interp(Vpeak+ref_dvaxis,vgrid,Qsm,left=np.nan,right=np.nan)
        if not np.isnan(refQ).any(): Vpeak0=Vpeak; ref_refQ=refQ; break
    if Vpeak0 is None: return None,None,"无参考峰",None
    rows=[]; n_total=0; fail=Counter()
    for f in aging:
        with fs.open(f) as fh:
            df=next(pq.ParquetFile(fh).iter_batches(batch_size=500000,
                    columns=['Zeit','Spannung','Strom','Zustand']).__iter__()).to_pandas()
        df['Zeit']=pd.to_datetime(df['Zeit'])
        for seg in get_raw(df,'DCH',-1):
            if not is_3c_dynamic(seg,cap_t,cap_v): continue
            n_total+=1
            feat,why=reconstruct(seg,ref_refQ,ref_dvaxis,Vpeak0)
            if feat is None: fail[why]+=1; continue
            t=pd.to_datetime(seg['Zeit'].iloc[0]); soh=float(pchip(t.value))
            row={'bid':bid,'Temp':tc['Temp'],'SOH':soh,'time':t,
                 'Vpeak':feat['Vpeak'],'Vpeak0':Vpeak0,'dVpeak':feat['dVpeak']}
            for k in range(N_PTS): row[f'Q_{k}']=feat['Q'][k]
            for k in range(N_PTS): row[f'dQ_{k}']=feat['dQ'][k]
            for k in range(N_PTS): row[f'IC_{k}']=feat['IC'][k]
            rows.append(row)
    return pd.DataFrame(rows), n_total, dict(fail), Vpeak0

# ===== 跑5块 =====
print(f"参数: HALF_WIN={HALF_WIN}, DV={DV}\n")
all_data=[]
for bid in VERIFY:
    try:
        d,ntot,fail,vp0=process(bid)
        if d is None: print(f"{bid}: {fail}"); continue
        d.to_csv(f"{bid}_peak_features.csv",index=False)
        rate=len(d)/ntot*100 if ntot else 0
        es=d[d['SOH']>92]
        ic8_e=es['IC_8'].corr(es['SOH']) if len(es)>10 else float('nan')
        print(f"{bid}: {len(d)}样本, 锚定率{rate:.1f}% ({len(d)}/{ntot}), "
              f"Vpeak0={vp0:.3f}, IC_8早期r={ic8_e:+.3f}, 失败{fail}")
        all_data.append(d)
    except Exception as e:
        print(f"{bid} 失败: {e}")

if all_data:
    full=pd.concat(all_data,ignore_index=True)
    print(f"\n5块合计{len(full)}样本")
    print(f"各块锚定率见上。重点:浅DOD(125)能不能锚定、早期样本是否补回")

参数: HALF_WIN=0.06, DV=0.002

008: 1569样本, 锚定率82.6% (1569/1900), Vpeak0=3.216, IC_8早期r=+0.962, 失败{'窗口超界': 330, 'vgrid不足': 1}
011: 1777样本, 锚定率97.1% (1777/1831), Vpeak0=3.200, IC_8早期r=+0.538, 失败{'窗口超界': 41, 'vgrid不足': 13}
012: 2651样本, 锚定率99.4% (2651/2668), Vpeak0=3.112, IC_8早期r=-0.709, 失败{'窗口超界': 17}
028: 916样本, 锚定率99.6% (916/920), Vpeak0=3.213, IC_8早期r=+0.956, 失败{'窗口超界': 3, 'vgrid不足': 1}
125 失败: time data "2025-10-06 15:03:59+00:00" doesn't match format "%Y-%m-%d %H:%M:%S.%f%z", at position 16. You might want to try:
    - passing `format` if your strings have a consistent format;
    - passing `format='ISO8601'` if your strings are all ISO8601 but not necessarily in exactly the same format;
    - passing `format='mixed'`, and the format will be inferred for each element individually. You might want to use `dayfirst` alongside this.

5块合计6913样本
各块锚定率见上。重点:浅DOD(125)能不能锚定、早期样本是否补回


In [ ]:
"""
第一部分(做法B): 提取 + EFC序列化 + 手工动态特征
"""
# ... 前面提取函数 extract_battery 与上一版完全相同(峰锚定,输出含EFC列) ...
# 这里假设已有 all_d = {bid: DataFrame}(每块电池的循环级特征,含EFC/SOH/IC_k/dQ_k/dVpeak)

# ===== EFC序列化 + 手工动态特征 =====
EFC_EARLY=100; N_SEQ=50; SOH_EOL=80
CURVE_COLS=[f'{p}_{k}' for p in ['Q','dQ','IC'] for k in range(16)]+['dVpeak']  # 49维序列特征
IC_COLS=[f'IC_{k}' for k in range(16)]
dQ_COLS=[f'dQ_{k}' for k in range(16)]

def dynamic_features(d_early):
    """d_early: 前100EFC的循环特征(已按EFC排序)。返回手工动态标量"""
    efc=d_early['EFC'].values
    # 1. IC峰高随EFC的斜率
    ic_peak=d_early[IC_COLS].abs().max(axis=1).values   # 每循环IC峰高
    ic_peak_slope=np.polyfit(efc,ic_peak,1)[0] if len(efc)>2 else 0.0
    # 2. dVpeak随EFC的斜率
    dvpeak_slope=np.polyfit(efc,d_early['dVpeak'].values,1)[0] if len(efc)>2 else 0.0
    # 3. ΔIC方差: 用前100EFC内 最晚IC曲线 - 最早IC曲线,取方差
    ic_early=d_early[IC_COLS].iloc[0].values
    ic_late=d_early[IC_COLS].iloc[-1].values
    dIC_var=np.var(ic_late-ic_early)
    # 4. dQ方差随EFC的斜率
    dQ_var_per_cycle=d_early[dQ_COLS].var(axis=1).values
    dQ_var_slope=np.polyfit(efc,dQ_var_per_cycle,1)[0] if len(efc)>2 else 0.0
    return {'ic_peak_slope':ic_peak_slope,'dvpeak_slope':dvpeak_slope,
            'dIC_var':dIC_var,'dQ_var_slope':dQ_var_slope}

print(f"=== EFC序列化(前{EFC_EARLY}EFC采{N_SEQ}点)+手工动态特征 ===")
X_seq=[]; X_dyn=[]; Y_life=[]; Vpeak0_list=[]; bids=[]
efc_grid=np.linspace(0,EFC_EARLY,N_SEQ)
DYN_COLS=['ic_peak_slope','dvpeak_slope','dIC_var','dQ_var_slope']

for bid,d in all_d.items():
    d=d.sort_values('EFC').reset_index(drop=True)
    if d['EFC'].max()<EFC_EARLY:
        print(f"{bid}: EFC不足{EFC_EARLY},跳过"); continue
    # --- 序列特征:efc_grid每点取最近循环 ---
    seq=np.zeros((N_SEQ,len(CURVE_COLS)))
    for i,e in enumerate(efc_grid):
        j=(d['EFC']-e).abs().idxmin()
        seq[i]=d.loc[j,CURVE_COLS].values
    # --- 手工动态特征:前100EFC内所有循环算 ---
    d_early=d[d['EFC']<=EFC_EARLY]
    dyn=dynamic_features(d_early)
    # --- 寿命 ---
    below=d[d['SOH']<=SOH_EOL]
    life=below['EFC'].iloc[0] if len(below)>0 else d['EFC'].max()
    X_seq.append(seq); X_dyn.append([dyn[c] for c in DYN_COLS])
    Y_life.append(life); Vpeak0_list.append(d['Vpeak0'].iloc[0]); bids.append(bid)
    print(f"{bid}: 寿命{life:.0f}EFC | IC峰斜率{dyn['ic_peak_slope']:+.4f} "
          f"dVpeak斜率{dyn['dvpeak_slope']:+.5f} ΔIC方差{dyn['dIC_var']:.3f} "
          f"dQ方差斜率{dyn['dQ_var_slope']:+.2e}")

X_seq=np.array(X_seq); X_dyn=np.array(X_dyn); Y_life=np.array(Y_life)
Vpeak0_arr=np.array(Vpeak0_list)
np.savez("seq_data_5.npz",X_seq=X_seq,X_dyn=X_dyn,Y=Y_life,
         Vpeak0=Vpeak0_arr,bids=np.array(bids),dyn_cols=np.array(DYN_COLS))
print(f"\n序列X{X_seq.shape}, 动态特征X{X_dyn.shape}, 寿命Y{Y_life.shape}")

# 看动态特征和寿命的关系(5块太少,只看趋势)
print(f"\n=== 动态特征 vs 寿命(5块,看方向) ===")
for i,c in enumerate(DYN_COLS):
    if len(Y_life)>=3:
        r=np.corrcoef(X_dyn[:,i],Y_life)[0,1]
        print(f"  {c}: r={r:+.3f}")

=== EFC序列化(前100EFC采50点)+手工动态特征 ===
008: 寿命627EFC | IC峰斜率-0.0230 dVpeak斜率+0.00002 ΔIC方差3.117 dQ方差斜率+6.71e-06
011: 寿命1099EFC | IC峰斜率+0.0010 dVpeak斜率+0.00002 ΔIC方差0.014 dQ方差斜率+1.12e-08
012: 寿命534EFC | IC峰斜率-0.0003 dVpeak斜率+0.00002 ΔIC方差0.003 dQ方差斜率-4.87e-10
028: 寿命726EFC | IC峰斜率+0.0015 dVpeak斜率+0.00003 ΔIC方差0.113 dQ方差斜率+1.65e-07
125: 寿命642EFC | IC峰斜率-0.0001 dVpeak斜率+0.00002 ΔIC方差0.001 dQ方差斜率-1.11e-08

序列X(5, 50, 49), 动态特征X(5, 4), 寿命Y(5,)

=== 动态特征 vs 寿命(5块,看方向) ===
  ic_peak_slope: r=+0.293
  dvpeak_slope: r=+0.168
  dIC_var: r=-0.249
  dQ_var_slope: r=-0.250


In [ ]:
"""
段1: 提取一块"新电池"的前100EFC数据(测试用)
复用第一部分的提取函数,但只取前100EFC
"""
# === 复用第一部分所有函数: TS/parse_cond/build_pchip_soh/build_cap_interp/
#     get_raw/is_3c/get_ic_peak/reconstruct/extract_battery/dynamic_features ===
# (此处省略,与第一部分相同)

TEST_BID='009'   # 选一块没参与训练的(009也是35/70/60,但没在训练5块里)

def extract_early(bid, efc_limit=100):
    """提取某电池,但只到efc_limit EFC"""
    d=extract_battery(bid)    # 完整提取(复用)
    if d is None: return None,None
    d=d.sort_values('EFC').reset_index(drop=True)
    d_early=d[d['EFC']<=efc_limit]
    if len(d_early)<10 or d_early['EFC'].max()<efc_limit*0.9:
        print(f"{bid} 前{efc_limit}EFC数据不足"); return None,None
    return d, d_early

print(f"=== 提取测试电池 {TEST_BID} 的前100EFC ===")
d_full, d_early = extract_early(TEST_BID, 100)
if d_early is not None:
    # 序列化(同训练)
    efc_grid=np.linspace(0,100,50)
    CURVE_COLS=[f'{p}_{k}' for p in ['Q','dQ','IC'] for k in range(16)]+['dVpeak']
    seq=np.zeros((50,len(CURVE_COLS)))
    for i,e in enumerate(efc_grid):
        j=(d_early['EFC']-e).abs().idxmin()
        seq[i]=d_early.loc[j,CURVE_COLS].values
    dyn=dynamic_features(d_early)
    DYN_COLS=['ic_peak_slope','dvpeak_slope','dIC_var','dQ_var_slope']
    test_seq=seq[None]                                    # [1,50,49]
    test_dyn=np.array([[dyn[c] for c in DYN_COLS]])       # [1,4]
    # 真实寿命(从完整数据,只为对比,预测时不用)
    below=d_full[d_full['SOH']<=80]
    true_life=below['EFC'].iloc[0] if len(below)>0 else d_full['EFC'].max()
    print(f"{TEST_BID} 前100EFC提取完成, 真实寿命(到SOH80)={true_life:.0f}EFC")
    np.savez("test_battery.npz",seq=test_seq,dyn=test_dyn,true_life=true_life,bid=TEST_BID)

=== 提取测试电池 009 的前100EFC ===


ValueError: not enough values to unpack (expected 4, got 2)

In [27]:
"""
单块验证 峰锚定提取 —— 无SOC/DOD版
只跑一块,验证:锚定率、Vpeak0、dVpeak随老化、特征与SOH相关性
输出不含SOC/DOD,只有Temp
修复了工业数据中 Timestamp 毫秒级缺失导致的解析崩溃问题
"""
import s3fs, re, numpy as np, pandas as pd
import pyarrow.parquet as pq
from scipy.ndimage import gaussian_filter1d
from scipy.signal import find_peaks
from scipy.interpolate import PchipInterpolator
import warnings; warnings.filterwarnings('ignore')
from s3_credentials import S3_ACCESS_KEY, S3_SECRET_KEY

OPTS={"key":S3_ACCESS_KEY,"secret":S3_SECRET_KEY,
      "client_kwargs":{"endpoint_url":"https://iseadocker.isea.rwth-aachen.de:9000","region_name":"us-east-1"},
      "config_kwargs":{"s3":{"addressing_style":"path"},"signature_version":"s3v4"}}
RAWBASE="projects/j8005-metabatt/Metabatt/A123"
CAP="projects/j8005-metabatt/Metabatt/A123/40_capacity_monitore"
fs=s3fs.S3FileSystem(**OPTS)

BID="214"
N_PTS=16; HALF_WIN=0.06; DV=0.005; SIGMA=2; TOL=0.25; PROM=0.15

def parse_temp_crate(af):
    for f in af:
        m=re.search(r'(\d+)grad_(\d+)SOC_(\d+)DOD_(\d+)C',f.split('/')[-1])
        if m: return {'Temp':int(m.group(1)),'C_Rate':int(m.group(4))}
    return None

def build_pchip_soh(bid):
    c=pd.read_csv(f"s3://{CAP}/METABatt_A123_APR18650M1B_{bid}_capacity.csv",
                  storage_options=OPTS).dropna(subset=['SOH'])
    # 核心修复：加入 format='ISO8601' 免疫时间戳格式截断问题
    c['t']=pd.to_datetime(c['CAP_start_time'], format='ISO8601'); c=c.sort_values('t').reset_index(drop=True)
    t_ns=c['t'].astype('int64').values.astype(float); soh=c['SOH'].values.copy()
    for i in range(1,len(soh)):
        if soh[i]>soh[i-1]: soh[i]=soh[i-1]
    return PchipInterpolator(t_ns,soh,extrapolate=True), c['Capacity_py'].iloc[0]

def build_cap_interp(bid):
    c=pd.read_csv(f"s3://{CAP}/METABatt_A123_APR18650M1B_{bid}_capacity.csv",
                  storage_options=OPTS).dropna(subset=['Capacity_py'])
    # 核心修复：加入 format='ISO8601'
    c['t']=pd.to_datetime(c['CAP_start_time'], format='ISO8601'); c=c.sort_values('t')
    return c['t'].astype('int64').values.astype(float), c['Capacity_py'].values

def get_raw(df,state,sign):
    m=((df['Zustand']==state)&(np.sign(df['Strom'])==sign)).values
    ch=np.diff(m.astype(int));st=np.where(ch==1)[0]+1;en=np.where(ch==-1)[0]+1
    if len(m)>0 and m[0]:st=np.r_[0,st]
    if len(en)<len(st):en=np.r_[en,len(df)]
    return [df.iloc[s:e] for s,e in zip(st,en) if e-s>=30]

def is_3c_dynamic(seg,cap_t,cap_v):
    t=pd.to_datetime(seg['Zeit'].iloc[0], format='ISO8601').value
    C_now=np.interp(t,cap_t,cap_v); It=3*C_now
    I=abs(seg['Strom'].mean())
    # 核心修复：加入 format='ISO8601'
    tt=(pd.to_datetime(seg['Zeit'], format='ISO8601')-pd.to_datetime(seg['Zeit'].iloc[0], format='ISO8601')).dt.total_seconds().values
    return (It*(1-TOL)<I<It*(1+TOL)) and tt[-1]>=60

def get_ic_peak(seg):
    V=seg['Spannung'].values; I=seg['Strom'].values
    # 核心修复：加入 format='ISO8601'
    tt=(pd.to_datetime(seg['Zeit'], format='ISO8601')-pd.to_datetime(seg['Zeit'].iloc[0], format='ISO8601')).dt.total_seconds().values
    Q=np.cumsum(np.abs(I)*np.diff(tt,prepend=tt[0])/3600.0)
    if Q[-1]<=0: return None
    o=np.argsort(V); Vu,idx=np.unique(V[o],return_index=True); Qu=Q[o][idx]
    if len(Vu)<10: return None
    vgrid=np.arange(Vu.min(),Vu.max(),DV)
    if len(vgrid)<2*N_PTS: return None
    Qsm=gaussian_filter1d(np.interp(vgrid,Vu,Qu),sigma=SIGMA)
    IC=np.abs(np.gradient(Qsm,vgrid))
    if IC.max()<=0: return None
    peaks,_=find_peaks(IC,prominence=IC.max()*PROM)
    if len(peaks)==0: return None
    Vpeak=vgrid[peaks[np.argmax(IC[peaks])]]
    return vgrid,Qsm,IC,Vpeak

def reconstruct(seg,ref_refQ,ref_dvaxis,Vpeak0):
    r=get_ic_peak(seg)
    if r is None: return None
    vgrid,Qsm,IC,Vpeak=r
    dv=np.linspace(-HALF_WIN,HALF_WIN,N_PTS); absv=Vpeak+dv
    Q16=np.interp(absv,vgrid,Qsm,left=np.nan,right=np.nan)
    IC16=np.interp(absv,vgrid,IC,left=np.nan,right=np.nan)
    if np.isnan(Q16).any(): return None
    dQ16=Q16-np.interp(dv,ref_dvaxis,ref_refQ) if ref_refQ is not None else np.zeros(N_PTS)
    return {'Q':Q16,'dQ':dQ16,'IC':IC16,'Vpeak':Vpeak,'dVpeak':Vpeak0-Vpeak}

# ===== 主流程 =====
BASE=f"{RAWBASE}/METABatt_A123_APR18650M1B_{BID}"
aging=sorted([f for f in fs.ls(BASE) if 'Aging' in f and 'Cyc' in f])
tc=parse_temp_crate(aging)
pchip,C0=build_pchip_soh(BID); cap_t,cap_v=build_cap_interp(BID)
print(f"电池{BID}: Temp={tc['Temp']}℃, C0={C0:.4f} (不读取SOC/DOD)")

# 第一遍:Vpeak0
Vpeak0=None; ref_refQ=None; ref_dvaxis=np.linspace(-HALF_WIN,HALF_WIN,N_PTS)
with fs.open(aging[0]) as fh:
    df0=next(pq.ParquetFile(fh).iter_batches(batch_size=500000,
            columns=['Zeit','Spannung','Strom','Zustand']).__iter__()).to_pandas()
df0['Zeit']=pd.to_datetime(df0['Zeit'], format='ISO8601')
for seg in get_raw(df0,'DCH',-1):
    if not is_3c_dynamic(seg,cap_t,cap_v): continue
    r=get_ic_peak(seg)
    if r is None: continue
    vgrid,Qsm,IC,Vpeak=r
    refQ=np.interp(Vpeak+ref_dvaxis,vgrid,Qsm,left=np.nan,right=np.nan)
    if not np.isnan(refQ).any():
        Vpeak0=Vpeak; ref_refQ=refQ; break
print(f"参考峰位 Vpeak0={Vpeak0:.4f}V")

"""
单块验证 峰锚定提取 —— 无SOC/DOD版
只跑一块,验证:锚定率、Vpeak0、dVpeak随老化、特征与SOH相关性
输出不含SOC/DOD,只有Temp
修复了工业数据中 Timestamp 毫秒级缺失导致的解析崩溃问题
"""
import s3fs, re, numpy as np, pandas as pd
import pyarrow.parquet as pq
from scipy.ndimage import gaussian_filter1d
from scipy.signal import find_peaks
from scipy.interpolate import PchipInterpolator
import warnings; warnings.filterwarnings('ignore')
from s3_credentials import S3_ACCESS_KEY, S3_SECRET_KEY

OPTS={"key":S3_ACCESS_KEY,"secret":S3_SECRET_KEY,
      "client_kwargs":{"endpoint_url":"https://iseadocker.isea.rwth-aachen.de:9000","region_name":"us-east-1"},
      "config_kwargs":{"s3":{"addressing_style":"path"},"signature_version":"s3v4"}}
RAWBASE="projects/j8005-metabatt/Metabatt/A123"
CAP="projects/j8005-metabatt/Metabatt/A123/40_capacity_monitore"
fs=s3fs.S3FileSystem(**OPTS)

BID="214"
N_PTS=16; HALF_WIN=0.06; DV=0.005; SIGMA=2; TOL=0.25; PROM=0.15

def parse_temp_crate(af):
    for f in af:
        m=re.search(r'(\d+)grad_(\d+)SOC_(\d+)DOD_(\d+)C',f.split('/')[-1])
        if m: return {'Temp':int(m.group(1)),'C_Rate':int(m.group(4))}
    return None

def build_pchip_soh(bid):
    c=pd.read_csv(f"s3://{CAP}/METABatt_A123_APR18650M1B_{bid}_capacity.csv",
                  storage_options=OPTS).dropna(subset=['SOH'])
    # 核心修复：加入 format='ISO8601' 免疫时间戳格式截断问题
    c['t']=pd.to_datetime(c['CAP_start_time'], format='ISO8601'); c=c.sort_values('t').reset_index(drop=True)
    t_ns=c['t'].astype('int64').values.astype(float); soh=c['SOH'].values.copy()
    for i in range(1,len(soh)):
        if soh[i]>soh[i-1]: soh[i]=soh[i-1]
    return PchipInterpolator(t_ns,soh,extrapolate=True), c['Capacity_py'].iloc[0]

def build_cap_interp(bid):
    c=pd.read_csv(f"s3://{CAP}/METABatt_A123_APR18650M1B_{bid}_capacity.csv",
                  storage_options=OPTS).dropna(subset=['Capacity_py'])
    # 核心修复：加入 format='ISO8601'
    c['t']=pd.to_datetime(c['CAP_start_time'], format='ISO8601'); c=c.sort_values('t')
    return c['t'].astype('int64').values.astype(float), c['Capacity_py'].values

def get_raw(df,state,sign):
    m=((df['Zustand']==state)&(np.sign(df['Strom'])==sign)).values
    ch=np.diff(m.astype(int));st=np.where(ch==1)[0]+1;en=np.where(ch==-1)[0]+1
    if len(m)>0 and m[0]:st=np.r_[0,st]
    if len(en)<len(st):en=np.r_[en,len(df)]
    return [df.iloc[s:e] for s,e in zip(st,en) if e-s>=30]

def is_3c_dynamic(seg,cap_t,cap_v):
    t=pd.to_datetime(seg['Zeit'].iloc[0], format='ISO8601').value
    C_now=np.interp(t,cap_t,cap_v); It=3*C_now
    I=abs(seg['Strom'].mean())
    # 核心修复：加入 format='ISO8601'
    tt=(pd.to_datetime(seg['Zeit'], format='ISO8601')-pd.to_datetime(seg['Zeit'].iloc[0], format='ISO8601')).dt.total_seconds().values
    return (It*(1-TOL)<I<It*(1+TOL)) and tt[-1]>=60

def get_ic_peak(seg):
    V=seg['Spannung'].values; I=seg['Strom'].values
    # 核心修复：加入 format='ISO8601'
    tt=(pd.to_datetime(seg['Zeit'], format='ISO8601')-pd.to_datetime(seg['Zeit'].iloc[0], format='ISO8601')).dt.total_seconds().values
    Q=np.cumsum(np.abs(I)*np.diff(tt,prepend=tt[0])/3600.0)
    if Q[-1]<=0: return None
    o=np.argsort(V); Vu,idx=np.unique(V[o],return_index=True); Qu=Q[o][idx]
    if len(Vu)<10: return None
    vgrid=np.arange(Vu.min(),Vu.max(),DV)
    if len(vgrid)<2*N_PTS: return None
    Qsm=gaussian_filter1d(np.interp(vgrid,Vu,Qu),sigma=SIGMA)
    IC=np.abs(np.gradient(Qsm,vgrid))
    if IC.max()<=0: return None
    peaks,_=find_peaks(IC,prominence=IC.max()*PROM)
    if len(peaks)==0: return None
    Vpeak=vgrid[peaks[np.argmax(IC[peaks])]]
    return vgrid,Qsm,IC,Vpeak

def reconstruct(seg,ref_refQ,ref_dvaxis,Vpeak0):
    r=get_ic_peak(seg)
    if r is None: return None
    vgrid,Qsm,IC,Vpeak=r
    dv=np.linspace(-HALF_WIN,HALF_WIN,N_PTS); absv=Vpeak+dv
    Q16=np.interp(absv,vgrid,Qsm,left=np.nan,right=np.nan)
    IC16=np.interp(absv,vgrid,IC,left=np.nan,right=np.nan)
    if np.isnan(Q16).any(): return None
    dQ16=Q16-np.interp(dv,ref_dvaxis,ref_refQ) if ref_refQ is not None else np.zeros(N_PTS)
    return {'Q':Q16,'dQ':dQ16,'IC':IC16,'Vpeak':Vpeak,'dVpeak':Vpeak0-Vpeak}

# ===== 主流程 =====
BASE=f"{RAWBASE}/METABatt_A123_APR18650M1B_{BID}"
aging=sorted([f for f in fs.ls(BASE) if 'Aging' in f and 'Cyc' in f])
tc=parse_temp_crate(aging)
pchip,C0=build_pchip_soh(BID); cap_t,cap_v=build_cap_interp(BID)
print(f"电池{BID}: Temp={tc['Temp']}℃, C0={C0:.4f} (不读取SOC/DOD)")

# 第一遍:Vpeak0
Vpeak0=None; ref_refQ=None; ref_dvaxis=np.linspace(-HALF_WIN,HALF_WIN,N_PTS)
with fs.open(aging[0]) as fh:
    df0=next(pq.ParquetFile(fh).iter_batches(batch_size=500000,
            columns=['Zeit','Spannung','Strom','Zustand']).__iter__()).to_pandas()
df0['Zeit']=pd.to_datetime(df0['Zeit'], format='ISO8601')
for seg in get_raw(df0,'DCH',-1):
    if not is_3c_dynamic(seg,cap_t,cap_v): continue
    r=get_ic_peak(seg)
    if r is None: continue
    vgrid,Qsm,IC,Vpeak=r
    refQ=np.interp(Vpeak+ref_dvaxis,vgrid,Qsm,left=np.nan,right=np.nan)
    if not np.isnan(refQ).any():
        Vpeak0=Vpeak; ref_refQ=refQ; break
print(f"参考峰位 Vpeak0={Vpeak0:.4f}V")

# 第二遍
rows=[]; n_total=0; n_anchor=0
for fi,f in enumerate(aging):
    with fs.open(f) as fh:
        df=next(pq.ParquetFile(fh).iter_batches(batch_size=500000,
                columns=['Zeit','Spannung','Strom','Zustand']).__iter__()).to_pandas()
    df['Zeit']=pd.to_datetime(df['Zeit'], format='ISO8601')
    for seg in get_raw(df,'DCH',-1):
        if not is_3c_dynamic(seg,cap_t,cap_v): continue
        n_total+=1
        feat=reconstruct(seg,ref_refQ,ref_dvaxis,Vpeak0)
        if feat is None: continue
        n_anchor+=1
        t=pd.to_datetime(seg['Zeit'].iloc[0], format='ISO8601')
        soh=float(pchip(t.value))
        row={'bid':BID,'Temp':tc['Temp'],'SOH':soh,'time':t,
             'Vpeak':feat['Vpeak'],'Vpeak0':Vpeak0,'dVpeak':feat['dVpeak']}
        for k in range(N_PTS): row[f'Q_{k}']=feat['Q'][k]
        for k in range(N_PTS): row[f'dQ_{k}']=feat['dQ'][k]
        for k in range(N_PTS): row[f'IC_{k}']=feat['IC'][k]
        rows.append(row)
    print(f"  [{fi+1}/{len(aging)}] 累计样本{len(rows)}")

data=pd.DataFrame(rows)
data.to_csv(f"{BID}_peak_features.csv",index=False)
print(f"\n✅ {len(data)}样本 → {BID}_peak_features.csv")
print(f"锚定成功率: {n_anchor}/{n_total} = {n_anchor/max(n_total,1)*100:.1f}%")
print(f"列: {[c for c in data.columns if not c.startswith(('Q_','dQ_','IC_'))]} + Q/dQ/IC各16 (无SOC/DOD)")

# 验证
d=data.sort_values('SOH',ascending=False)
print(f"\n=== 验证 ===")
print(f"Vpeak随老化: 早期{d['Vpeak'].head(50).mean():.4f} → 晚期{d['Vpeak'].tail(50).mean():.4f}V "
      f"(横移{(d['Vpeak'].head(50).mean()-d['Vpeak'].tail(50).mean())*1000:+.1f}mV)")
print(f"dVpeak随老化: 早期{d['dVpeak'].head(50).mean():+.4f} → 晚期{d['dVpeak'].tail(50).mean():+.4f}")
print(f"\n特征与SOH相关性(全程/早期SOH>92):")
es=data[data['SOH']>92]
for col in ['dVpeak','Vpeak','IC_8','Q_8','dQ_8']:
    rg=data[col].corr(data['SOH'])
    re_=es[col].corr(es['SOH']) if len(es)>10 else float('nan')
    print(f"  {col:8s}: 全程{rg:+.3f} 早期{re_:+.3f}")
# 第二遍
rows=[]; n_total=0; n_anchor=0
for fi,f in enumerate(aging):
    with fs.open(f) as fh:
        df=next(pq.ParquetFile(fh).iter_batches(batch_size=500000,
                columns=['Zeit','Spannung','Strom','Zustand']).__iter__()).to_pandas()
    df['Zeit']=pd.to_datetime(df['Zeit'], format='ISO8601')
    for seg in get_raw(df,'DCH',-1):
        if not is_3c_dynamic(seg,cap_t,cap_v): continue
        n_total+=1
        feat=reconstruct(seg,ref_refQ,ref_dvaxis,Vpeak0)
        if feat is None: continue
        n_anchor+=1
        t=pd.to_datetime(seg['Zeit'].iloc[0], format='ISO8601')
        soh=float(pchip(t.value))
        row={'bid':BID,'Temp':tc['Temp'],'SOH':soh,'time':t,
             'Vpeak':feat['Vpeak'],'Vpeak0':Vpeak0,'dVpeak':feat['dVpeak']}
        for k in range(N_PTS): row[f'Q_{k}']=feat['Q'][k]
        for k in range(N_PTS): row[f'dQ_{k}']=feat['dQ'][k]
        for k in range(N_PTS): row[f'IC_{k}']=feat['IC'][k]
        rows.append(row)
    print(f"  [{fi+1}/{len(aging)}] 累计样本{len(rows)}")

data=pd.DataFrame(rows)
data.to_csv(f"{BID}_peak_features.csv",index=False)
print(f"\n✅ {len(data)}样本 → {BID}_peak_features.csv")
print(f"锚定成功率: {n_anchor}/{n_total} = {n_anchor/max(n_total,1)*100:.1f}%")
print(f"列: {[c for c in data.columns if not c.startswith(('Q_','dQ_','IC_'))]} + Q/dQ/IC各16 (无SOC/DOD)")

# 验证
d=data.sort_values('SOH',ascending=False)
print(f"\n=== 验证 ===")
print(f"Vpeak随老化: 早期{d['Vpeak'].head(50).mean():.4f} → 晚期{d['Vpeak'].tail(50).mean():.4f}V "
      f"(横移{(d['Vpeak'].head(50).mean()-d['Vpeak'].tail(50).mean())*1000:+.1f}mV)")
print(f"dVpeak随老化: 早期{d['dVpeak'].head(50).mean():+.4f} → 晚期{d['dVpeak'].tail(50).mean():+.4f}")
print(f"\n特征与SOH相关性(全程/早期SOH>92):")
es=data[data['SOH']>92]
for col in ['dVpeak','Vpeak','IC_8','Q_8','dQ_8']:
    rg=data[col].corr(data['SOH'])
    re_=es[col].corr(es['SOH']) if len(es)>10 else float('nan')
    print(f"  {col:8s}: 全程{rg:+.3f} 早期{re_:+.3f}")

电池214: Temp=45℃, C0=1.1582 (不读取SOC/DOD)


TypeError: unsupported format string passed to NoneType.__format__